# Bob the Butler — A30 fine-tuning notebook

Fine-tune a pretrained SmolVLM with LoRA, rather than training from scratch. This notebook prepares **single-step GUI pointing and chat-to-search-query extraction**. It does not by itself teach reliable multi-step desktop control.

Run all cells in your A30 Jupyter environment. If you have no dataset yet, the notebook generates a synthetic starter set automatically; replace it with labeled real workflows before production use. It has not been executed on your A30 yet. Keep the working CUDA-enabled PyTorch installation already on that machine. Start with controlled test-app screenshots and separate train/evaluation workflows.


In [ ]:
%pip install "transformers==4.57.6" "peft==0.18.0" "accelerate==1.12.0" pillow


In [ ]:
import json, random, math
from pathlib import Path
import torch
from PIL import Image
from transformers import AutoProcessor, AutoModelForImageTextToText, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model

assert torch.cuda.is_available(), "Select the A30 CUDA kernel before continuing."
print(torch.cuda.get_device_name(0), torch.cuda.get_device_properties(0).total_memory / 2**30, "GiB VRAM")
MODEL_ID = "HuggingFaceTB/SmolVLM-500M-Instruct"
REVISION = "a7da5b986cb59b408707209984f360a5f4ad7e47"
# Smaller alternative: HuggingFaceTB/SmolVLM-256M-Instruct
# Its pinned revision: 7e3e67edbbed1bf9888184d9df282b700a323964
DATA_ROOT = Path("bob-data")
DATASET = DATA_ROOT / "steps.jsonl"
OUTPUT = Path("bob-smolvlm-lora")
DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
random.seed(42)
torch.manual_seed(42)


## Dataset

Create `bob-data/steps.jsonl`, one object per line. Image paths are relative to `bob-data`. Coordinates are normalized to the image, not raw desktop pixels. Use many layouts, sizes and controls; the two synthetic benchmark images are insufficient training data.

```json
{"type":"grounding","workflow_id":"editor-save-001","image":"images/save.png","instruction":"Click Save","target":{"x":0.5,"y":0.6},"target_bbox":[0.45,0.55,0.55,0.65]}
{"type":"intent","workflow_id":"search-nvidia-001","instruction":"hey can you open stocks for nvidai","target":{"query":"NVIDIA stock"}}
```

Keep related screenshots/typos within one `workflow_id`. The split below holds out entire workflows, avoiding nearly identical screenshots on both sides. Add ask/stop examples and richer task histories in a subsequent training stage.

The next cell generates a reproducible synthetic starter set if no dataset exists. It lets you run the notebook immediately, but it is only a controlled bootstrap exercise: success on synthetic buttons does not establish accuracy in Chrome or other real applications. Replace it with recorded, labeled workflows before production use.


In [ ]:
# Optional starter data: generated locally on your A30 environment; no uploads.
# Existing bob-data/steps.jsonl is never overwritten.
if not DATASET.exists():
    from PIL import ImageDraw, ImageFont
    (DATA_ROOT / "images").mkdir(parents=True, exist_ok=True)
    rng = random.Random(42)
    labels = ["Open", "Save", "Cancel", "Search", "Next", "Back", "Help", "Settings", "Download", "Done"]
    generated = []
    def get_font(size):
        try: return ImageFont.truetype("DejaVuSans.ttf", size)
        except OSError: return ImageFont.load_default(size=size)
    # Hold out complete appearance families, not neighboring images from one workflow.
    for family in range(8):
        for index in range(40):
            image = Image.new("RGB", (640, 360), ["#edf2f8", "#f5efe8", "#edf5ed", "#eef0fa"][family % 4])
            draw = ImageDraw.Draw(image)
            draw.rounded_rectangle((25, 25, 615, 335), radius=12, fill="white", outline="#9eafbf", width=2)
            draw.text((45, 45), f"Bob test window {family}", font=get_font(20), fill="#36465b")
            names = rng.sample(labels, 4)
            columns = rng.sample([45, 190, 335, 480], 4)
            target_index = rng.randrange(4)
            target_box = None
            for j, (name, x) in enumerate(zip(names, columns)):
                y = rng.choice([120, 210, 270])
                box = (x, y, x + 112, y + 46)
                draw.rounded_rectangle(box, radius=7, fill=["#e7eef8", "#e8f3ea", "#f6eada"][family % 3], outline="#98acc2", width=1)
                draw.text((x + 56, y + 23), name, anchor="mm", font=get_font(16), fill="#30465f")
                if j == target_index: target_box = box
            name = f"images/family-{family}-{index}.png"
            image.save(DATA_ROOT / name)
            x1, y1, x2, y2 = target_box
            generated.append({"type":"grounding", "workflow_id":f"synthetic-family-{family}", "image":name,
                              "instruction":rng.choice([f"Click {names[target_index]}", f"Point to the {names[target_index]} button"]),
                              "target":{"x":round((x1+x2)/1280, 4), "y":round((y1+y2)/720, 4)},
                              "target_bbox":[x1/640,y1/360,x2/640,y2/360]})
    companies = {"NVIDIA":["nvidai","nvidia","NVDA"], "Apple":["apple","appl","AAPL"],
                 "Microsoft":["microsoft","microsft","MSFT"], "Tesla":["tesla","tesal","TSLA"],
                 "Amazon":["amazon","amzon","AMZN"], "AMD":["amd","AMD"],
                 "Intel":["intel","intell","INTC"], "Netflix":["netflix","netflx","NFLX"]}
    templates = ["hey can you open stocks for {}", "show me {} stock", "search for {} stock price", "open the stock page for {}"]
    for company, spellings in companies.items():
        for spelling in spellings:
            for template in templates:
                generated.append({"type":"intent", "workflow_id":f"intent-company-{company}",
                                  "instruction":template.format(spelling), "target":{"query":f"{company} stock"}})
    DATASET.write_text("\n".join(json.dumps(row) for row in generated) + "\n")
    print("Created", len(generated), "synthetic starter examples. These do not replace real desktop demonstrations.")
else:
    print("Using your existing dataset:", DATASET)


In [ ]:
rows = [json.loads(line) for line in DATASET.read_text().splitlines() if line.strip()]
for row in rows:
    assert row["type"] in ("grounding", "intent") and row.get("workflow_id")
    if row["type"] == "grounding":
        assert (DATA_ROOT / row["image"]).is_file()
        assert all(type(row["target"][k]) in (int, float) and math.isfinite(row["target"][k]) and 0 <= row["target"][k] <= 1 for k in ("x", "y"))
    else:
        assert isinstance(row["target"]["query"], str)
groups = sorted({row["workflow_id"] for row in rows})
assert len(groups) >= 5 and len(rows) >= 20, "Collect at least 20 examples across five workflows for a smoke training run; meaningful evaluation needs considerably more."
random.shuffle(groups)
eval_groups = set(groups[:max(1, len(groups)//5)])
train_rows = [row for row in rows if row["workflow_id"] not in eval_groups]
eval_rows = [row for row in rows if row["workflow_id"] in eval_groups]
print("Train/evaluation:", len(train_rows), len(eval_rows))


In [ ]:
processor = AutoProcessor.from_pretrained(MODEL_ID, revision=REVISION)
base = AutoModelForImageTextToText.from_pretrained(MODEL_ID, revision=REVISION, torch_dtype=DTYPE).to("cuda")
# Adapt only text-model Q/V projections; leave the vision encoder/projector frozen.
targets = [name for name, module in base.named_modules()
           if isinstance(module, torch.nn.Linear)
           and ("text_model" in name or "language_model" in name)
           and name.endswith(("q_proj", "v_proj"))]
assert targets, "Inspect this model's module names before choosing LoRA targets."
model = get_peft_model(base, LoraConfig(r=8, lora_alpha=16, lora_dropout=0.05, target_modules=targets, task_type="CAUSAL_LM"))
model.print_trainable_parameters()


In [ ]:
GROUND_SYSTEM = "Identify the requested UI target. Return one JSON object with numeric x and y coordinates normalized from 0 to 1. Do not invent extra controls."
INTENT_SYSTEM = "Turn the user request into a web search query. Correct obvious spelling mistakes. Return only a JSON object with a query string. Do not give stock prices."

def prepare(row, include_target):
    image = None
    content = []
    if row["type"] == "grounding":
        image = Image.open(DATA_ROOT / row["image"]).convert("RGB")
        image.thumbnail((640, 480))
        content.append({"type":"image"})
    content.append({"type":"text", "text":row["instruction"]})
    messages = [{"role":"system", "content":[{"type":"text", "text":GROUND_SYSTEM if image is not None else INTENT_SYSTEM}]},
                {"role":"user", "content":content}]
    prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_args = {"images":[[image]]} if image is not None else {}
    prompt_inputs = processor(text=[prompt], return_tensors="pt", **image_args)
    for key, value in prompt_inputs.items():
        if value.is_floating_point(): prompt_inputs[key] = value.to(DTYPE)
    if not include_target:
        return prompt_inputs
    full = processor.apply_chat_template(messages + [{"role":"assistant", "content":[{"type":"text", "text":json.dumps(row["target"], separators=(",", ":"))}]}], tokenize=False, add_generation_prompt=False)
    inputs = processor(text=[full], return_tensors="pt", **image_args)
    assert inputs["input_ids"].shape[1] <= 2048, "Reduce image tiling/prompt length for the deployment context limit."
    labels = inputs["input_ids"].clone()
    labels[:, :prompt_inputs["input_ids"].shape[1]] = -100
    labels[inputs["attention_mask"] == 0] = -100
    assert (labels != -100).any(), "No assistant tokens remained after prompt masking."
    inputs["labels"] = labels
    for key, value in inputs.items():
        if value.is_floating_point(): inputs[key] = value.to(DTYPE)
    return inputs

class Rows(torch.utils.data.Dataset):
    def __init__(self, values): self.values = values
    def __len__(self): return len(self.values)
    def __getitem__(self, index): return self.values[index]

def collate(batch):
    assert len(batch) == 1  # Variable image/text shapes; accumulate gradients instead.
    return prepare(batch[0], include_target=True)


In [ ]:
args = TrainingArguments(
    output_dir=str(OUTPUT), num_train_epochs=2, learning_rate=1e-4,
    per_device_train_batch_size=1, per_device_eval_batch_size=1, gradient_accumulation_steps=8,
    bf16=DTYPE == torch.bfloat16, fp16=DTYPE == torch.float16,
    gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant":False},
    remove_unused_columns=False, dataloader_num_workers=0,
    logging_steps=5, eval_strategy="epoch", save_strategy="epoch", report_to="none",
)
trainer = Trainer(model=model, args=args, train_dataset=Rows(train_rows), eval_dataset=Rows(eval_rows), data_collator=collate)
training_run = trainer.train()
print(training_run.metrics)
model.save_pretrained(OUTPUT)
processor.save_pretrained(OUTPUT)


## Evaluate generated answers, not just training loss

The generation call below uses only the held-out prompt and image: the target answer is not supplied to the model. Treat these as basic checks, and evaluate full desktop tasks separately before enabling Auto model control.


In [ ]:
model.eval()
results = []
for row in eval_rows:
    inputs = {key:value.to("cuda") for key,value in prepare(row, include_target=False).items()}
    with torch.inference_mode():
        output = model.generate(**inputs, max_new_tokens=64, do_sample=False, use_cache=True)
    text = processor.batch_decode(output[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)[0]
    hit = False
    try:
        prediction = json.loads(text)
        if row["type"] == "grounding":
            x, y = prediction["x"], prediction["y"]
            box = row.get("target_bbox")
            hit = (type(x) in (int,float) and type(y) in (int,float) and box is not None
                   and box[0] <= x <= box[2] and box[1] <= y <= box[3])
        else:
            hit = prediction["query"].strip().casefold() == row["target"]["query"].strip().casefold()
    except (ValueError, KeyError, TypeError, AttributeError): pass
    results.append({"workflow_id":row["workflow_id"], "type":row["type"], "output":text, "hit":bool(hit)})
print("Held-out smoke accuracy:", sum(r["hit"] for r in results), "/", len(results))
(OUTPUT / "heldout-results.json").write_text(json.dumps(results, indent=2))

metrics_by_type = {}
for kind in ("grounding", "intent"):
    subset = [record for record in results if record["type"] == kind]
    metrics_by_type[kind] = {"correct":sum(record["hit"] for record in subset), "total":len(subset)}
    print(kind, metrics_by_type[kind])
summary = {"model":MODEL_ID, "revision":REVISION, "gpu":torch.cuda.get_device_name(0),
           "train_examples":len(train_rows), "evaluation_examples":len(eval_rows),
           "train_metrics":training_run.metrics, "generated_answer_results":metrics_by_type,
           "peak_gpu_allocated_gib":torch.cuda.max_memory_allocated()/2**30,
           "evaluation_scope":"Starter/single-step evaluation only; not a full desktop-agent benchmark."}
(OUTPUT / "run_summary.json").write_text(json.dumps(summary, indent=2, default=str))


In [ ]:
# Export merged weights after reviewing evaluation results.
EXPORT = Path("bob-smolvlm-merged")
merged = model.merge_and_unload()
merged.save_pretrained(EXPORT, safe_serialization=True)
processor.save_pretrained(EXPORT)
print("Merged model:", EXPORT.resolve())


## Deploy to UNO Q

Convert the merged Hugging Face model to GGUF with the pinned llama.cpp converter, then quantize and benchmark it on the board. Keep or reconvert the matching multimodal projector; the vision stack was frozen here. Verify conversion support and compare outputs before deployment.

Do not copy the training environment or A30 dependencies onto the Q. Deploy only the inference runtime, model/projector, and receiver adapter. Re-run the same board benchmark after quantization. A successful notebook run is not evidence that the smaller board meets the latency or accuracy requirements.

Send back `bob-smolvlm-lora/run_summary.json` and `bob-smolvlm-lora/heldout-results.json`, plus any error traceback. Keep the merged weights on your A30 environment for now. We will check the results before converting and transferring the model.

The stock-search example also requires a browser/text-entry controller. This notebook teaches query extraction and single-step pointing; it does not implement browser launching, typing, or full task planning.
